In [ ]:

import pandas as pd
import numpy as np

def wilder_rma(series: pd.Series, length: int) -> pd.Series:
    """Wilder's RMA used by Pine ta.atr() → EMA with alpha = 1/length."""
    if length <= 1:
        return series
    return series.ewm(alpha=1/length, adjust=False).mean()

def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low  - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

def weis_wave_volume(
    df: pd.DataFrame,
    methodtype: str = "ATR",                 # "ATR" | "Traditional" | "Part of Price"
    methodvalue: float = 14.0,               # box size or ATR length or divisor for Part-of-Price
    pricesource: str = "Close",              # "Close" | "Open / Close" | "High / Low"
    useTrueRange: str = "Auto",              # "Always" | "Auto" | "Never"
    isOscillating: bool = False,             # if True, down waves plotted negative
    normalize: bool = False                  # if True, divide accumulated vol by barcount
) -> pd.DataFrame:
    """
    Reproduces the TradingView WWV script logic in Python.
    df index should be datetime; columns: open, high, low, close, volume.
    """

    # --- Choose price source (replicates Pine logic) ---
    useClose = (pricesource == "Close")
    useOpenClose = (pricesource == "Open / Close") or useClose
    op = df["close"] if useClose else df["open"]
    if useOpenClose:
        hi = np.maximum(df["close"], op)
        lo = np.minimum(df["close"], op)
    else:
        hi, lo = df["high"], df["low"]

    # --- True Range and "volume" series selection (Always/Auto/Never) ---
    tr = true_range(df["high"], df["low"], df["close"])
    if useTrueRange == "Always":
        vol_series = tr
    elif useTrueRange == "Never":
        vol_series = df["volume"]
    else:  # Auto: use TR when volume is NA, otherwise volume
        vol_series = df["volume"].where(df["volume"].notna(), tr)

    # --- Box size (methodtype): ATR | Traditional | Part of Price ---
    if methodtype == "ATR":
        length = max(1, int(round(methodvalue)))
        mv = wilder_rma(tr, length)                   # box size per bar
    elif methodtype == "Part of Price":
        mv = df["close"] / float(methodvalue)         # box size per bar
    else:  # Traditional → fixed box
        mv = pd.Series(float(methodvalue), index=df.index)

    # --- Stateful wave calculation (matches Pine’s renko-like assignment) ---
    n = len(df)
    currclose   = pd.Series(index=df.index, dtype=float)
    direction   = pd.Series(index=df.index, dtype=int)   # +1 up, -1 down, 0 flat init
    barcount    = pd.Series(index=df.index, dtype=int)
    wave_vol    = pd.Series(index=df.index, dtype=float)
    res         = pd.Series(index=df.index, dtype=float)

    for i in range(n):
        if i == 0:
            currclose.iloc[i] = df["close"].iloc[i]
            direction.iloc[i] = 0
            barcount.iloc[i]  = 1
            wave_vol.iloc[i]  = vol_series.iloc[i]
        else:
            prevc = currclose.iloc[i-1]
            prevdir = direction.iloc[i-1]
            prev_barcount = barcount.iloc[i-1]
            prev_wave_vol = wave_vol.iloc[i-1]

            prevhigh = prevc + mv.iloc[i]
            prevlow  = prevc - mv.iloc[i]

            cclose = hi.iloc[i] if hi.iloc[i] > prevhigh else (
                     lo.iloc[i] if lo.iloc[i] < prevlow else prevc)
            currclose.iloc[i] = cclose

            if cclose > prevc:
                direction.iloc[i] = 1
            elif cclose < prevc:
                direction.iloc[i] = -1
            else:
                direction.iloc[i] = prevdir

            direction_has_changed = (direction.iloc[i] != prevdir)

            # barcount := !directionHasChanged && normalize ? prev + 1 : 1
            barcount.iloc[i] = (prev_barcount + 1) if (normalize and not direction_has_changed) else 1

            # vol := !directionHasChanged ? prev + vol : vol
            wave_vol.iloc[i] = (prev_wave_vol + vol_series.iloc[i]) if not direction_has_changed else vol_series.iloc[i]

        res.iloc[i] = (wave_vol.iloc[i] / barcount.iloc[i]) if (barcount.iloc[i] > 1) else wave_vol.iloc[i]

    # Oscillating: flip sign for down waves
    if isOscillating:
        res = res.where(direction >= 0, -res)

    out = pd.DataFrame({"wwv": res, "direction": direction})
    out.index.name = df.index.name or "date"
    return out


In [ ]:

import plotly.graph_objects as go
import numpy as np

# Example params (mirror your Pine inputs)
params = dict(
    methodtype   = "ATR",          # "ATR" | "Traditional" | "Part of Price"
    methodvalue  = 14.0,
    pricesource  = "Open / Close", # "Close" | "Open / Close" | "High / Low"
    useTrueRange = "Auto",         # "Always" | "Auto" | "Never"
    isOscillating= False,
    normalize    = False
)

wwv = weis_wave_volume(df, **params)  # df must have open/high/low/close/volume

# Color columns: green for up, red for down (alpha ~75 in Pine)
colors = np.where(wwv["direction"] >= 0, "rgba(0,128,0,0.75)", "rgba(220,20,60,0.75)")

fig = go.Figure()
fig.add_trace(go.Bar(x=wwv.index, y=wwv["wwv"], marker_color=colors, name="Weis Wave Volume"))
fig.update_layout(
    title="Weis Wave Volume (Bloomberg BQuant)",
    yaxis_title="Wave Volume",
    bargap=0.0,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()


In [ ]:

from bql import Service
import pandas as pd

bq = Service()

SEC = "NG1 COMDTY"        # replace with your futures/spot index ticker
START = "-2y"             # BQL range expression, e.g., -2y to 0d

query = f"""
get(
    OPEN(dates=range({START},0d)),
    HIGH(dates=range({START},0d)),
    LOW(dates=range({START},0d)),
    PX_LAST(dates=range({START},0d)),
    VOLUME(dates=range({START},0d))
) for(['{SEC}'])
"""

res = bq.exec(query)
df = res[0].df().reset_index()
df = df.rename(columns={"Date":"date", "OPEN":"open", "HIGH":"high", "LOW":"low", "PX_LAST":"close", "VOLUME":"volume"})
df = df.set_index("date").sort_index()
